# N033 · 二维前缀和与容斥

**目标：** 用四项容斥查询矩形和且统一坐标约定。

**先修：** N018, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 二维累加；边界补零；半开矩形；容斥。

## 从问题到算法

二维前缀同时复用行方向与列方向的累计和，但左上重叠区被算了两次，必须减去一次。为边界补零行和零列后，所有矩形查询都能统一为四项容斥，避免为贴边区域写特殊分支。所有query参数采用半开区间。

## 最小实现

**本章接口：** `PrefixSum2D`、`matrix_block_sum(mat, k)`

In [1]:
class PrefixSum2D:
    def __init__(self,mat):
        self.rows=len(mat); self.cols=len(mat[0]) if mat else 0
        self.p=[[0]*(self.cols+1) for _ in range(self.rows+1)]
        for i in range(self.rows):
            for j in range(self.cols):
                self.p[i+1][j+1]=mat[i][j]+self.p[i][j+1]+self.p[i+1][j]-self.p[i][j]
    def query(self,r1,c1,r2,c2):
        if not (0<=r1<=r2<=self.rows and 0<=c1<=c2<=self.cols): raise IndexError('invalid rectangle')
        p=self.p
        return p[r2][c2]-p[r1][c2]-p[r2][c1]+p[r1][c1]

def matrix_block_sum(mat,k):
    p=PrefixSum2D(mat)
    return [[p.query(max(0,i-k),max(0,j-k),min(p.rows,i+k+1),min(p.cols,j+k+1))
             for j in range(p.cols)] for i in range(p.rows)]

## 正确性、前提与复杂度

用集合容斥看四个前缀：目标外的上方与左方各减一次，它们共同左上区域被减两次，再加回一次。

**代价：** 构建 O(RC) 时间与空间；查询 O(1)；整张block sum输出 O(RC)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

m=[[1,2],[3,4]]; p=PrefixSum2D(m)
show_table(['前缀表行','内容'],list(enumerate(p.p)))
show_table(['半开矩形','查询和'], [((r1,c1,r2,c2),p.query(r1,c1,r2,c2)) for r1,c1,r2,c2 in [(0,0,2,2),(1,1,2,2),(0,1,2,2)]])

前缀表行,内容
0,"[0, 0, 0]"
1,"[0, 1, 3]"
2,"[0, 4, 10]"


半开矩形,查询和
"(0, 0, 2, 2)",10
"(1, 1, 2, 2)",4
"(0, 1, 2, 2)",6


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
p=PrefixSum2D([[1,2],[3,4]])
assert p.query(0,0,2,2)==10 and p.query(1,1,2,2)==4
assert matrix_block_sum([[1,2],[3,4]],3)==[[10,10],[10,10]]
from random import Random
rng=Random(33)
for rows in range(1,5):
    for cols in range(1,5):
        m=[[rng.randrange(-3,4) for _ in range(cols)] for _ in range(rows)]
        p=PrefixSum2D(m)
        for a in range(rows+1):
            for b in range(a,rows+1):
                for c in range(cols+1):
                    for d in range(c,cols+1):
                        assert p.query(a,c,b,d)==sum(m[i][j] for i in range(a,b) for j in range(c,d))
print('N033: 所有本章断言通过')

N033: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 不存额外边界行列时比较代码复杂度。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 扩展统计矩形内非零数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 304. Range Sum Query 2D - Immutable（canonical）
- 1314. Matrix Block Sum（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。